In [1]:
from pathlib import Path
from zipfile import ZipFile
import json
import re
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

PASTA_DADOS = Path("dados")
PASTA_BRUTOS = PASTA_DADOS / "brutos"
PASTA_TRATADOS = PASTA_DADOS / "tratados"

PASTA_BRUTOS.mkdir(parents=True, exist_ok=True)
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)

In [2]:
municipios_abc = pd.DataFrame({
    "codigo_ibge": [
        "3513801", "3529401", "3543303", "3544103",
        "3547809", "3548708", "3548807"
    ],
    "municipio": [
        "Diadema", "Mauá", "Ribeirão Pires", "Rio Grande da Serra",
        "Santo André", "São Bernardo do Campo", "São Caetano do Sul"
    ],
    "uf": ["SP"] * 7
})

municipios_abc

,codigo_ibge,municipio,uf
0,3513801,Diadema,SP
1,3529401,Mauá,SP
2,3543303,Ribeirão Pires,SP
3,3544103,Rio Grande da Serra,SP
4,3547809,Santo André,SP
5,3548708,São Bernardo do Campo,SP
6,3548807,São Caetano do Sul,SP


In [3]:
url_sidra = (
    "https://apisidra.ibge.gov.br/values/t/4709/"
    "n6/3513801,3529401,3543303,3544103,3547809,3548708,3548807/"
    "v/93/p/2022"
)

resposta = requests.get(url_sidra, timeout=60)
resposta.raise_for_status()
dados_sidra = resposta.json()

# A primeira linha da resposta contém os nomes das colunas.
populacao_ibge = pd.DataFrame(dados_sidra[1:])

populacao_ibge.head()

,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,6,Município,45,Pessoas,393237,3513801,Diadema (SP),93,População residente,2022,2022
1,6,Município,45,Pessoas,418261,3529401,Mauá (SP),93,População residente,2022,2022
2,6,Município,45,Pessoas,115559,3543303,Ribeirão Pires (SP),93,População residente,2022,2022
3,6,Município,45,Pessoas,44170,3544103,Rio Grande da Serra (SP),93,População residente,2022,2022
4,6,Município,45,Pessoas,748919,3547809,Santo André (SP),93,População residente,2022,2022


In [4]:
populacao_ibge = populacao_ibge.rename(columns={
    "D1C": "codigo_ibge",
    "D1N": "municipio_ibge",
    "V": "populacao_2022"
})

populacao_ibge["codigo_ibge"] = populacao_ibge["codigo_ibge"].astype(str)
populacao_ibge["populacao_2022"] = pd.to_numeric(
    populacao_ibge["populacao_2022"], errors="coerce"
)

populacao_ibge = populacao_ibge[
    ["codigo_ibge", "municipio_ibge", "populacao_2022"]
].drop_duplicates()

populacao_ibge

,codigo_ibge,municipio_ibge,populacao_2022
0,3513801,Diadema (SP),393237
1,3529401,Mauá (SP),418261
2,3543303,Ribeirão Pires (SP),115559
3,3544103,Rio Grande da Serra (SP),44170
4,3547809,Santo André (SP),748919
5,3548708,São Bernardo do Campo (SP),810729
6,3548807,São Caetano do Sul (SP),165655


In [5]:
# Conferência: esperamos exatamente os sete municípios.
conferencia_ibge = municipios_abc.merge(
    populacao_ibge,
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

display(conferencia_ibge)

assert len(conferencia_ibge) == 7, "A consulta não retornou os sete municípios."
assert conferencia_ibge["populacao_2022"].notna().all(), (
    "Há município sem população. Confira a resposta da API SIDRA."
)

populacao_ibge.to_csv(
    PASTA_TRATADOS / "populacao_abc_censo_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

,codigo_ibge,municipio,uf,municipio_ibge,populacao_2022
0,3513801,Diadema,SP,Diadema (SP),393237
1,3529401,Mauá,SP,Mauá (SP),418261
2,3543303,Ribeirão Pires,SP,Ribeirão Pires (SP),115559
3,3544103,Rio Grande da Serra,SP,Rio Grande da Serra (SP),44170
4,3547809,Santo André,SP,Santo André (SP),748919
5,3548708,São Bernardo do Campo,SP,São Bernardo do Campo (SP),810729
6,3548807,São Caetano do Sul,SP,São Caetano do Sul (SP),165655


In [6]:
url_cnes = (
    "https://s3.sa-east-1.amazonaws.com/"
    "ckan.saude.gov.br/CNES/cnes_estabelecimentos_json.zip"
)

arquivo_zip_cnes = PASTA_BRUTOS / "cnes_estabelecimentos_json.zip"

if not arquivo_zip_cnes.exists():
    with requests.get(url_cnes, stream=True, timeout=180) as resposta:
        resposta.raise_for_status()
        with open(arquivo_zip_cnes, "wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    arquivo.write(bloco)

print(f"Arquivo salvo em: {arquivo_zip_cnes}")
print(f"Tamanho: {arquivo_zip_cnes.stat().st_size / (1024**2):.1f} MB")

Arquivo salvo em: dados\brutos\cnes_estabelecimentos_json.zip
Tamanho: 65.4 MB


In [7]:
with ZipFile(arquivo_zip_cnes) as z:
    arquivos_json = [nome for nome in z.namelist() if nome.lower().endswith(".json")]
    print("Arquivos JSON encontrados:", arquivos_json[:10])
    print("Total de arquivos JSON:", len(arquivos_json))

    if not arquivos_json:
        raise ValueError("Não foi encontrado arquivo JSON dentro do ZIP.")

    nome_json = arquivos_json[0]
    with z.open(nome_json) as arquivo:
        conteudo_json = json.load(arquivo)

print("Tipo do conteúdo:", type(conteudo_json).__name__)
if isinstance(conteudo_json, dict):
    print("Chaves principais:", list(conteudo_json.keys())[:30])
elif isinstance(conteudo_json, list):
    print("Quantidade de registros:", len(conteudo_json))
    print("Exemplo de registro:", conteudo_json[0] if conteudo_json else "Lista vazia")

Arquivos JSON encontrados: ['cnes_estabelecimentos.json']
Total de arquivos JSON: 1
Tipo do conteúdo: list
Quantidade de registros: 642608
Exemplo de registro: {'CO_CNES': '19', 'CO_UNIDADE': '2602900000019', 'CO_UF': '26', 'CO_IBGE': '260290', 'NU_CNPJ_MANTENEDORA': '11294402000162', 'NO_RAZAO_SOCIAL': 'PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO', 'NO_FANTASIA': 'POLICLINICA DR JAMACI DE MEDEIROS', 'CO_NATUREZA_ORGANIZACAO': '', 'DS_NATUREZA_ORGANIZACAO': '', 'TP_GESTAO': 'M', 'CO_NIVEL_HIERARQUIA': '', 'DS_NIVEL_HIERARQUIA': '', 'CO_ESFERA_ADMINISTRATIVA': 'M ', 'DS_ESFERA_ADMINISTRATIVA': 'MUNICIPAL', 'CO_ATIVIDADE': '04', 'TP_UNIDADE': '4', 'CO_CEP': '54580530', 'NO_LOGRADOURO': 'RUA 21 DE ABRIL', 'NU_ENDERECO': 'S/N', 'NO_BAIRRO': 'PONTE DOS CARVALHOS', 'NU_TELEFONE': '(81)35221848', 'NU_LATITUDE': '-8.23166', 'NU_LONGITUDE': '-34.9695991', 'CO_TURNO_ATENDIMENTO': '06', 'DS_TURNO_ATENDIMENTO': 'ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:INCLUI SABADOS, DOMINGOS E FERIADOS)

In [8]:
def localizar_lista_de_registros(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        # Procura uma lista de dicionários nas chaves mais comuns.
        for chave in ["estabelecimentos", "data", "dados", "results", "resultados"]:
            valor = obj.get(chave)
            if isinstance(valor, list):
                return valor
        # Procura qualquer valor que seja uma lista de dicionários.
        for valor in obj.values():
            if isinstance(valor, list) and (not valor or isinstance(valor[0], dict)):
                return valor
    raise ValueError(
        "Não foi possível identificar a lista de registros. "
        "Inspecione conteudo_json e ajuste esta função."
    )

registros_cnes = localizar_lista_de_registros(conteudo_json)
cnes_bruto = pd.json_normalize(registros_cnes, sep="_")

print("Dimensão:", cnes_bruto.shape)
print("Colunas disponíveis:")
print(cnes_bruto.columns.tolist())

display(cnes_bruto.head(3))

Dimensão: (642608, 36)
Colunas disponíveis:
['CO_CNES', 'CO_UNIDADE', 'CO_UF', 'CO_IBGE', 'NU_CNPJ_MANTENEDORA', 'NO_RAZAO_SOCIAL', 'NO_FANTASIA', 'CO_NATUREZA_ORGANIZACAO', 'DS_NATUREZA_ORGANIZACAO', 'TP_GESTAO', 'CO_NIVEL_HIERARQUIA', 'DS_NIVEL_HIERARQUIA', 'CO_ESFERA_ADMINISTRATIVA', 'DS_ESFERA_ADMINISTRATIVA', 'CO_ATIVIDADE', 'TP_UNIDADE', 'CO_CEP', 'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_BAIRRO', 'NU_TELEFONE', 'NU_LATITUDE', 'NU_LONGITUDE', 'CO_TURNO_ATENDIMENTO', 'DS_TURNO_ATENDIMENTO', 'NU_CNPJ', 'NO_EMAIL', 'CO_NATUREZA_JUR', 'ST_CENTRO_CIRURGICO', 'ST_CENTRO_OBSTETRICO', 'ST_CENTRO_NEONATAL', 'ST_ATEND_HOSPITALAR', 'ST_SERVICO_APOIO', 'ST_ATEND_AMBULATORIAL', 'CO_MOTIVO_DESAB', 'CO_AMBULATORIAL_SUS']


,CO_CNES,CO_UNIDADE,CO_UF,CO_IBGE,NU_CNPJ_MANTENEDORA,NO_RAZAO_SOCIAL,NO_FANTASIA,CO_NATUREZA_ORGANIZACAO,DS_NATUREZA_ORGANIZACAO,TP_GESTAO,CO_NIVEL_HIERARQUIA,DS_NIVEL_HIERARQUIA,CO_ESFERA_ADMINISTRATIVA,DS_ESFERA_ADMINISTRATIVA,CO_ATIVIDADE,TP_UNIDADE,CO_CEP,NO_LOGRADOURO,NU_ENDERECO,NO_BAIRRO,NU_TELEFONE,NU_LATITUDE,NU_LONGITUDE,CO_TURNO_ATENDIMENTO,DS_TURNO_ATENDIMENTO,NU_CNPJ,NO_EMAIL,CO_NATUREZA_JUR,ST_CENTRO_CIRURGICO,ST_CENTRO_OBSTETRICO,ST_CENTRO_NEONATAL,ST_ATEND_HOSPITALAR,ST_SERVICO_APOIO,ST_ATEND_AMBULATORIAL,CO_MOTIVO_DESAB,CO_AMBULATORIAL_SUS
0,19,2602900000019,26,260290,11294402000162,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA DR JAMACI DE MEDEIROS,,,M,,,M,MUNICIPAL,04,4,54580530,RUA 21 DE ABRIL,S/N,PONTE DOS CARVALHOS,(81)35221848,-8.23166,-34.9695991,06,"ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:INCLUI SABADOS, DOMINGOS E FERIADOS)",,,1244,0.0,0.0,0.0,0.0,1.0,0.0,,SIM
1,27,2602900000027,26,260290,10930451000181,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,CASA DE SAUDE SANTA HELENA,,,M,,,M,MUNICIPAL,04,5,54505560,AVN PRESIDENTE GETULIO VARGAS,428,CENTRO,(81)35210355,-8.2801925142,-35.0314371459,06,"ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:INCLUI SABADOS, DOMINGOS E FERIADOS)",10930451001153,,2062,1.0,1.0,1.0,1.0,1.0,1.0,,SIM
2,35,2602900000035,26,260290,11294402000162,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,HOSPITAL MENDO SAMPAIO,,,M,,,M,MUNICIPAL,04,5,54535430,BR 101 SUL KM 33,S/N,CHARNECA,(81)35210857,-8.29221434,-35.0481980206,06,"ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:INCLUI SABADOS, DOMINGOS E FERIADOS)",11294402000243,,1244,1.0,1.0,1.0,1.0,1.0,1.0,,SIM


In [9]:
candidatos_codigo_municipio = [
    "codigo_municipio", "CO_MUNICIPIO_GESTOR", "CO_MUNICIPIO",
    "codigo_ibge", "cod_municipio", "municipio_codigo",
    "codigo_municipio_gestor"
]

colunas_normalizadas = {str(col).lower(): col for col in cnes_bruto.columns}
coluna_codigo_cnes = None

for candidato in candidatos_codigo_municipio:
    if candidato.lower() in colunas_normalizadas:
        coluna_codigo_cnes = colunas_normalizadas[candidato.lower()]
        break

print("Coluna candidata encontrada:", coluna_codigo_cnes)

# Mostra colunas cujo nome contém 'mun' ou 'ibge' para facilitar a conferência.
[col for col in cnes_bruto.columns
 if ("mun" in str(col).lower()) or ("ibge" in str(col).lower())]

Coluna candidata encontrada: None


['CO_IBGE']

In [14]:
coluna_codigo_cnes = "CO_IBGE"

print("Coluna identificadora:", coluna_codigo_cnes)

Coluna identificadora: CO_IBGE


In [15]:
print("Tipo de dado:", cnes_bruto["CO_IBGE"].dtype)

display(cnes_bruto[["CO_IBGE"]].head(10))

print("Quantidade de municípios encontrados:", 
      cnes_bruto["CO_IBGE"].nunique())

Tipo de dado: object


,CO_IBGE
0,260290
1,260290
2,260290
3,260290
4,260290
5,260290
6,260290
7,260290
8,260290
9,260290


Quantidade de municípios encontrados: 5585


In [16]:
cnes_bruto["CO_IBGE"] = (
    cnes_bruto["CO_IBGE"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(7)
)

municipios["codigo_ibge"] = (
    municipios["codigo_ibge"]
    .astype("string")
    .str.zfill(7)
)

NameError: name 'municipios' is not defined